In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

In [29]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [30]:
from src.cleaning import load
from src.cleaning import config

In [31]:

pp = load.read_price_paid(config.DATA_PATH, config.PP_FILENAME)
pp.shape[0]

135557

In [32]:
epc = load.read_epc(config.DATA_PATH, config.EPC_FILENAME, config.EPC_COLS)
epc.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

Correct row number and dtypes

In [33]:
epc = load.filter_local_authorities(epc, config.TARGET_LOCAL_AUTHORITIES)

epc.shape

(216050, 28)

4 rows dropped as expected

In [34]:
from src.cleaning import normalise

epc = normalise.strip_whitespace(epc)
epc = normalise.remove_commas(epc, config.EPC_ADDRESS_COLS)
epc['address1']

Commas replaced:
address1    83515.0
address2     3910.0
address3      137.0
postcode        0.0
dtype: float64


0                  10 Newman Road
1                 10 Barlow Close
2                65 Campbell Road
3           12 St. Michaels Close
4         Flat 10 Canalside House
                   ...           
216049               30 Appletons
216050        1d Walton Well Road
216051            8 Sycamore Road
216052        68 Spencer Crescent
216053             21 Ochre Close
Name: address1, Length: 216050, dtype: str

In [40]:
epc = normalise.build_address(epc, config.EPC_ADDRESS_COLS)

In [41]:
epc = normalise.build_tokens(epc)

epc['address_tokens']

0                   frozenset({ROAD, 3UJ, 10, OX4, NEWMAN})
1         frozenset({CLOSE, WHEATLEY, BARLOW, 10, 1NL, O...
2                 frozenset({ROAD, 3PG, 65, OX4, CAMPBELL})
3         frozenset({MICHAELS, CLOSE, ST., 6BE, SHIPTON-...
4         frozenset({ROAD, TRAMWAY, FLAT, 10, HOUSE, 5BG...
                                ...                        
216049                frozenset({7GG, OX12, 30, APPLETONS})
216050        frozenset({6ED, ROAD, WALTON, WELL, OX2, 1D})
216051             frozenset({ROAD, OX2, 9EJ, 8, SYCAMORE})
216052         frozenset({68, CRESCENT, SPENCER, OX4, 4SW})
216053    frozenset({CLOSE, OCHRE, WHEATLEY, 1FQ, OX33, ...
Name: address_tokens, Length: 216050, dtype: object